In [16]:
import numpy as np
import pandas as pd
RESULTS_CSV = "./initial_data/traditional_iso16_fullpilot_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = "./initial_data/traditional_iso16_fullpilot_AQA_combined_grid_search_diagnostics.csv"
FINAL_COLUMNS = [] #What columns we wish to keep

df1 = pd.read_csv(RESULTS_CSV)
df2 = pd.read_csv(DIAGNOSTICS_CSV)

print(df2.columns)

#Not including total cost because of raw terms and not decided on weights
needed_results_cols = ["Simulator_RSR_Total_Cost", "P_actual_watts", 
                       "R_actual_watts", "Pbar_ratio", "R_ratio",
                       "server_count", "utilization", "workload_mix_size",
                       "workload_mix"]



results_df = df1[needed_results_cols].copy()
results_df = results_df.rename(columns={"Simulator_RSR_Total_Cost": "M_RSR", 
                                "P_actual_watts": "P_actual", 
                                "R_actual_watts": "R_actual"})
results_df["C_track"] = df2["Ctrack_Weighted_Cost"]
results_df["C_Qos"] = df2["Diagnostic_FlexDC_SoftPlus_QoS_Cost"]
results_df["C_total"] = df2["Diagnostic_FullPaperObjective_Cost"]




weight_cols = sorted(
    [col for col in df1.columns if col.startswith("Weight_") and col != "Weight_Sample_ID"],
    key=lambda col: int(col.split("_")[-1])
)

# One array per row, e.g. shape: (number_of_rows, number_of_weights)
weights = df1[weight_cols].to_numpy()

import ast

results_df["workload_mix"] = [
    [
        job + [weight]
        for job, weight in zip(
            ast.literal_eval(workload_mix),
            weight_row.tolist()
        )
    ]
    for workload_mix, weight_row in zip(results_df["workload_mix"], weights)
]


print(weights)


print(df1.head())
print(df2.head())


### Now we organize the results to match the older condor framework order

print(results_df.columns)

results_df = results_df[[
    "C_total",
    "M_RSR",
    "C_track",
    "C_Qos",
    "P_actual",
    "R_actual",
    "Pbar_ratio",
    "R_ratio",
    "server_count",
    "utilization",
    "workload_mix_size",
    "workload_mix",
]]

results_df.to_csv("flexdc_all_data.csv", index=False)

Index(['Source_Output_Dir', 'Source_CSV_Path', 'Iteration', 'Weight_Sample_ID',
       'Workload_Name', 'Workload_Config', 'Pbar_kw_per_server',
       'R_kw_per_server', 'P_actual_watts', 'R_actual_watts', 'Pbar_ratio',
       'R_ratio', 'server_count', 'utilization', 'PR_Sweep_Mode',
       'Pmin_kw_per_server', 'Pmax_kw_per_server',
       'Pbar_lower_bound_kw_per_server', 'Pbar_upper_bound_kw_per_server',
       'PR_upper_bound_kw_per_server', 'R_lower_bound_kw_per_server',
       'R_max_for_this_Pbar_kw_per_server', 'P_plus_R_kw_per_server',
       'P_minus_R_kw_per_server', 'Pbar_points', 'R_points_per_Pbar',
       'PR_Total_Pairs_Before_Chunk', 'PR_Chunk_Index', 'PR_Num_Chunks',
       'PR_Chunk_Start_Index', 'PR_Chunk_End_Index_Exclusive',
       'PR_Pairs_In_Chunk', 'Weight_Equal_Value',
       'Weight_Relative_Lower_Bound', 'Weight_Relative_Upper_Bound',
       'Weight_Server_Lower_Bound', 'Weight_Upper_From_Lower_Bound',
       'Weight_Final_Lower_Bound', 'Weight_Final_Uppe